In [1]:
!pip install batchgenerators

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 4.5 MB/s eta 0:00:00
  Created wheel for batchgenerators: filename=batchgenerators-0.25.1-py3-none-any.whl size=93088 sha256=53d7ba1b42b55212ea73dcf3b2948e22afce07ad4c07ce362337c0a8bb4df97b
  Stored in directory: /root/.cache/pip/wheels/56/11/c7/fadca30e054c602093ffe36ba8a2f0a87dd2f86ac75191d3ed
Successfully built batchgenerators


In [1]:
!git clone https://github.com/mohammadnabia/DA_nnUNet.git
%cd DA_nnUNet
!pip install -e .

fatal: destination path 'DA_nnUNet' already exists and is not an empty directory.
/content/DA_nnUNet
Obtaining file:///content/DA_nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!unzip -q "/content/drive/MyDrive/BraTS-peds2023/BraTS-PEDs-2023.zip" -d /content/BraTS_PEDs_2023

In [3]:
import json
import os
import shutil

split_file = "/content/drive/MyDrive/100_epoch_Deepest_training_on_bratspeds_finetune/splits_final.json"
with open(split_file, 'r') as f:
    splits = json.load(f)

val_cases = splits[0]['val']  # Fold 0

source_dir = "/content/BraTS_PEDs_2023/ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData"
labels_dir = "/content/labelsTs_eval"
os.makedirs(labels_dir, exist_ok=True)

for case_id in val_cases:
    case_path = os.path.join(source_dir, case_id)
    seg_file = os.path.join(case_path, f"{case_id}-seg.nii.gz")
    if os.path.exists(seg_file):
        shutil.copy(seg_file, os.path.join(labels_dir, f"{case_id}.nii.gz"))
    else:
        print(f"Segmentation missing for case: {case_id}")


In [4]:
print("Number of segmentation files:", len(os.listdir(labels_dir)))


Number of segmentation files: 20


In [5]:
predictions_dir = "/content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_DeepestFT100epoch_80pruned_noTTA"


In [6]:
import SimpleITK as sitk

def compute_metrics(pred_path, gt_path):
    pred = sitk.ReadImage(pred_path)
    gt = sitk.ReadImage(gt_path)

    pred = sitk.Cast(pred, sitk.sitkUInt8)
    gt = sitk.Cast(gt, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(pred, gt)
    dice_score = dice_filter.GetDiceCoefficient()

    hausdorff_filter = sitk.HausdorffDistanceImageFilter()
    hausdorff_filter.Execute(pred, gt)
    hausdorff_distance = hausdorff_filter.GetHausdorffDistance()

    return dice_score, hausdorff_distance


In [7]:
import os
import SimpleITK as sitk
import numpy as np
import pandas as pd

# مسیر ground truth segmentation
gt_dir = "/content/labelsTs_eval"

# مسیر پیش‌بینی‌های pruned
pred_dir = "/content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_DeepestFT100epoch_80pruned_noTTA"

dice_list = []
hausdorff_list = []
case_names = []

for filename in os.listdir(gt_dir):
    if filename.endswith(".nii.gz"):
        gt_path = os.path.join(gt_dir, filename)
        pred_path = os.path.join(pred_dir, filename)
        if not os.path.exists(pred_path):
            print(f"Prediction missing for case: {filename}")
            continue

        dice, haus = compute_metrics(pred_path, gt_path)
        dice_list.append(dice)
        hausdorff_list.append(haus)
        case_names.append(filename)

        print(f"{filename} - Dice: {dice:.4f} - Hausdorff: {haus:.4f}")

# ذخیره نتایج در یک CSV
results = pd.DataFrame({
    "Case": case_names,
    "Dice": dice_list,
    "Hausdorff": hausdorff_list
})
results.to_csv("evaluation_results.csv", index=False)
print("Results saved to evaluation_results.csv")


BraTS-PED-00078-000.nii.gz - Dice: 0.9174 - Hausdorff: 56.7803
BraTS-PED-00096-000.nii.gz - Dice: 0.8643 - Hausdorff: 39.2046
BraTS-PED-00107-000.nii.gz - Dice: 0.9510 - Hausdorff: 3.6056
BraTS-PED-00026-000.nii.gz - Dice: 0.8612 - Hausdorff: 62.6817
BraTS-PED-00008-000.nii.gz - Dice: 0.8698 - Hausdorff: 7.6158
BraTS-PED-00104-000.nii.gz - Dice: 0.9544 - Hausdorff: 4.6904
BraTS-PED-00118-000.nii.gz - Dice: 0.8296 - Hausdorff: 6.4807
BraTS-PED-00042-000.nii.gz - Dice: 0.7423 - Hausdorff: 49.6991
BraTS-PED-00115-000.nii.gz - Dice: 0.2816 - Hausdorff: 26.7208
BraTS-PED-00132-000.nii.gz - Dice: 0.8393 - Hausdorff: 5.4772
BraTS-PED-00021-000.nii.gz - Dice: 0.8171 - Hausdorff: 13.4536
BraTS-PED-00050-000.nii.gz - Dice: 0.8211 - Hausdorff: 6.9282
BraTS-PED-00086-000.nii.gz - Dice: 0.8433 - Hausdorff: 70.7814
BraTS-PED-00055-000.nii.gz - Dice: 0.7999 - Hausdorff: 16.2788
BraTS-PED-00079-000.nii.gz - Dice: 0.9283 - Hausdorff: 55.8301
BraTS-PED-00099-000.nii.gz - Dice: 0.8788 - Hausdorff: 7.8740

In [8]:
import SimpleITK as sitk
import numpy as np

def compute_metrics_per_class(pred_path, gt_path, label_mapping):
    pred = sitk.ReadImage(pred_path)
    gt = sitk.ReadImage(gt_path)

    pred = sitk.GetArrayFromImage(pred)
    gt = sitk.GetArrayFromImage(gt)

    results = {}
    for class_name, labels in label_mapping.items():
        pred_mask = np.isin(pred, labels).astype(np.uint8)
        gt_mask = np.isin(gt, labels).astype(np.uint8)

        if np.sum(gt_mask) == 0 and np.sum(pred_mask) == 0:
            # هیچکدوم وجود ندارن
            dice_score = 1.0
            hausdorff_distance = 0.0
        elif np.sum(gt_mask) == 0 or np.sum(pred_mask) == 0:
            # یکی از دو مورد وجود نداره
            dice_score = 0.0
            hausdorff_distance = np.nan
        else:
            pred_mask = sitk.GetImageFromArray(pred_mask)
            pred_mask.CopyInformation(sitk.ReadImage(pred_path))
            gt_mask = sitk.GetImageFromArray(gt_mask)
            gt_mask.CopyInformation(sitk.ReadImage(gt_path))

            dice_filter = sitk.LabelOverlapMeasuresImageFilter()
            dice_filter.Execute(pred_mask, gt_mask)
            dice_score = dice_filter.GetDiceCoefficient()

            hausdorff_filter = sitk.HausdorffDistanceImageFilter()
            hausdorff_filter.Execute(pred_mask, gt_mask)
            hausdorff_distance = hausdorff_filter.GetHausdorffDistance()

        results[class_name] = (dice_score, hausdorff_distance)

    return results


In [9]:
import os
import pandas as pd

# دیکشنری برچسب‌ها (از dataset.json)
label_mapping = {
    "whole tumor": [1, 2, 3],
    "tumor core": [2, 3],
    "enhancing tumor": [3]
}

# آدرس‌ها
pred_folder = "/content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_DeepestFT100epoch_80pruned_noTTA"
gt_folder = "/content/labelsTs_eval"

# ساختار دیتافریم خروجی
results_list = []

# لیست کیس‌ها (بر اساس Ground Truth)
gt_cases = [f for f in os.listdir(gt_folder) if f.endswith('.nii.gz')]

for gt_file in gt_cases:
    case_id = gt_file.replace('.nii.gz', '')
    pred_file = case_id + ".nii.gz"   # تغییر این خط
    pred_path = os.path.join(pred_folder, pred_file)
    gt_path = os.path.join(gt_folder, gt_file)

    if not os.path.exists(pred_path):
        print(f"❌ پیش‌بینی برای {case_id} پیدا نشد.")
        continue

    metrics = compute_metrics_per_class(pred_path, gt_path, label_mapping)
    for class_name, (dice, hausdorff) in metrics.items():
        results_list.append({
            "case_id": case_id,
            "class": class_name,
            "dice": dice,
            "hausdorff": hausdorff
        })


# ساخت دیتافریم و ذخیره نتایج
df = pd.DataFrame(results_list)
output_path = "/content/evaluation_results_per_class.csv"
df.to_csv(output_path, index=False)
print(f"✅ نتایج ذخیره شد: {output_path}")


✅ نتایج ذخیره شد: /content/evaluation_results_per_class.csv


In [10]:
summary = df.groupby('class').agg({
    'dice': ['mean', 'median'],
    'hausdorff': ['mean', 'median']
}).reset_index()

# بازنویسی نام ستون‌ها برای نمایش
summary.columns = ['Class', 'Mean Dice', 'Median Dice', 'Mean HD95', 'Median HD95']
print(summary)


             Class  Mean Dice  Median Dice  Mean HD95  Median HD95
0  enhancing tumor   0.388404     0.374724  35.993797    25.618729
1       tumor core   0.379461     0.281617  34.625685    28.806709
2      whole tumor   0.897090     0.932521  27.760329    14.866222


In [11]:
from tabulate import tabulate

print(tabulate(summary, headers='keys', tablefmt='pretty', showindex=False))


+-----------------+--------------------+--------------------+--------------------+--------------------+
|      Class      |     Mean Dice      |    Median Dice     |     Mean HD95      |    Median HD95     |
+-----------------+--------------------+--------------------+--------------------+--------------------+
| enhancing tumor | 0.3884036925539459 | 0.3747241530551556 | 35.99379676399901  | 25.618729225374253 |
|   tumor core    | 0.3794606226270781 | 0.2816168641883985 |  34.6256850998964  | 28.806708762780847 |
|   whole tumor   | 0.8970902642996619 | 0.9325211506051656 | 27.760329458287192 | 14.866222321586708 |
+-----------------+--------------------+--------------------+--------------------+--------------------+


In [12]:
mean_dice_overall = np.nanmean(df['dice'])
mean_hausdorff_overall = np.nanmean(df['hausdorff'])

print(f"✅ میانگین کلی Dice: {mean_dice_overall:.4f}")
print(f"✅ میانگین کلی Hausdorff: {mean_hausdorff_overall:.4f}")


✅ میانگین کلی Dice: 0.5550
✅ میانگین کلی Hausdorff: 32.4883
